# Job 49840619 - the within-round deduction order was decoration

**Headline.** Retrained with the deductions inside each DD round **permuted** instead of sorted
by node index, the model scores **0.8393** on the shared 1500-instance pool against the
sorted-order baseline's 0.8540: delta -0.0147, 139 discordant instances won and 161 lost,
McNemar exact **p = 0.2253**. The ascending-index convention that `solve_with_dd_trace` emits
was available to the model and worth 0.632 to a hand rule with no deduction in it, but a model
trained without it solves the task just as well, so it was not the mechanism.

This is a **negative result and a falsified prediction of mine** (pre-registration (a) predicted
a resolvable fall). It is also a null, not a proof of no effect: the paired 95% interval on the
delta is about [-0.037, +0.008], so a cost of up to ~3.7 points is still inside it.

**Provenance**

| | |
|---|---|
| Slurm job | 49840619, `COMPLETED`, ExitCode 0:0, elapsed 05:50:50, ended 2026-10-27T19:27:37 |
| script | `scripts/slurm/sudoku2_n24_eventorder.sbatch` |
| git commit at submit | `77ebb35` (submitted 2026-10-27T13:31:42) |
| seed | 0 |
| instances | N=24, degree 5, q=4, rho 0.30-0.95, removal hints, 80000 train |
| the one change | `shuffle_events=True` (`shuffle_adj=True` in both arms) |
| architecture | d_model 256, 4 heads, L=12, ff=1024, pre-norm, lr=1e-3 constant, batch 32, 300ep |
| run dir | `$SCRATCH/spin_gnn/sudoku_n24_eventorder_2026-10-27` |
| checkpoint | `d0256_h04_L12_ff01024_lr1e-03_pre_shuf_ev` |
| deck | `notebooks/sudoku_eventorder_2026-10-28/eventorder.pdf` (9 pages) |

The comparison arm is finding 17's leak-free baseline (job 49659502), which differs from this
run in exactly one config field. It is row 0 of the shared-pool table below because
`sudoku_agg_eval` scored both checkpoints in the same call, on the same instances.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n24_eventorder_2026-10-27"
PREV = f"{SCRATCH}/spin_gnn/sudoku_n24_leakfree_2026-10-26"
pd.set_option("display.width", 200)

CELLS = ["sorted (finding 17)", "permuted (shuffle_events)"]
pool = pd.read_csv(f"{RUN}/sharedpool_n1500.csv")
pool["cell"] = CELLS   # row order = the order the checkpoints were passed, reference first
sweep = pd.concat([pd.read_csv(f"{PREV}/sweep.csv"), pd.read_csv(f"{RUN}/sweep.csv")],
                  ignore_index=True)
sweep["cell"] = CELLS
depth = {CELLS[0]: pd.read_csv(f"{PREV}/depth_n24_leakfree.csv"),
         CELLS[1]: pd.read_csv(f"{RUN}/depth_n24_eventorder.csv")}
om = pd.read_csv(f"{RUN}/ordering_and_memorization.csv")
print(len(pool), "pool rows;", len(sweep), "sweep rows;", len(om), "ordering/memorization rows")

## 1. Metric of record: the shared n=1500 pool

### Why two cells trained on different target orders can share a pool at all

`sudoku_agg_eval` prompts with `full_tokens[:reasoning_start]` -- hints and graph only -- and
scores *free generation* against the planted colouring through an order-insensitive
`check_solution`. The one thing `shuffle_events` changes, the order of events inside a round of
the **target** trace, lands entirely in the discarded suffix. So the two cells are handed
byte-identical prompts on the same instances and the paired McNemar test applies.

That is why `shuffle_events` is deliberately **not** in `INSTANCE_KEYS`, unlike `shuffle_adj`
which changes the prompt itself. Every *teacher-forced* analysis is the opposite case -- there
the flag changes the scored sequence -- and all nine of those call sites pass it through.

In [ ]:
cols = ["cell", "solve_rate", "wilson_lo", "wilson_hi", "delta",
        "discordant_won", "discordant_lost", "mcnemar_exact_p", "truncated_rate",
        "outcome_accuracy"]
display(pool[cols].round(4))

# The interval that matters for a null: how much cost does p=0.23 actually rule out?
r = pool.iloc[1]
half = 1.96 * np.sqrt(r.discordant_won + r.discordant_lost) / r.n_instances
print(f"paired 95% CI on delta: [{r.delta - half:+.4f}, {r.delta + half:+.4f}]")
print(f"-> rules out a cost larger than {abs(r.delta - half) * 100:.1f} points, not all cost")

In [ ]:
scols = ["cell", "n_params", "max_len", "final_val_nll", "final_val_accuracy",
         "mean_o1_solve_rate"]
display(sweep[scols].round(4))

# Measured on CPU before submitting (scripts/verify_shuffle_events.py, 300 instances):
# permuting a round of k simultaneous deductions injects log(k!) nats no model can predict.
FLOOR_NLL, FLOOR_ACC = 0.0270, -0.0163
d_nll = sweep.final_val_nll[1] - sweep.final_val_nll[0]
d_acc = sweep.final_val_accuracy[1] - sweep.final_val_accuracy[0]
print(f"val_nll  {d_nll:+.4f} vs irreducible floor {FLOOR_NLL:+.4f}")
print(f"val_acc  {d_acc:+.4f} vs irreducible floor {FLOOR_ACC:+.4f}")
print("-> one entropy floor away and no further: the permutation cost no predictive")
print("   capacity elsewhere in the sequence. n_params and max_len are identical, as")
print("   predicted: the flag cannot change the sequence LENGTH.")

### Why `val_nll` is only a check here and not the headline

Token metrics are **not comparable across this flag**, and that is measurable rather than a
hedge: the permuted targets carry log(k!) extra nats per k-event round that no model can
predict. Over `evaluate()`'s own denominator (every valid token, graph section included) that
floor is +0.0270 nats/token and -0.0163 token accuracy. The permuted arm lands at +0.0323 and
-0.0142 -- one floor away and no further. `solve_rate`, which is order-insensitive, is the
headline for exactly this reason.

## 2. The depth cost survives (pre-registration c), and the drop is bigger

`sudoku_depth_eval` bins a 600-instance pool by DD depth T at fixed *realized* density, using
the same generator stream as the baseline, so the bins are identical row by row.

In [ ]:
for rho in (0.60, 0.75):
    print(f"=== rho={rho}")
    t = pd.DataFrame({c: d[d.rho_requested == rho].set_index("T").solve_rate.round(4)
                      for c, d in depth.items()})
    t["n"] = depth[CELLS[0]].query("rho_requested == @rho").set_index("T").n
    display(t)

for c, d in depth.items():
    g = d[d.rho_requested == 0.60].sort_values("T")
    r_ = float(np.corrcoef(g["T"], g.solve_rate)[0, 1])
    print(f"{c:28s} r={r_:+.3f}  T=2..5 drop={g.solve_rate.iloc[0] - g.solve_rate.iloc[-1]:.3f}")
print()
print("Note the two statistics disagree in direction, which is why both are reported: the")
print("permuted arm's DROP is nearly twice as large, but its r is a hair LESS negative")
print("because r scores linearity as well as slope and the permuted curve bends.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for c, col in zip(CELLS, ["#1f77b4", "#d62728"]):
    g = depth[c].query("rho_requested == 0.60").sort_values("T")
    k = (g.solve_rate * g.n).round().astype(int)
    a, b = stats.beta.interval(0.95, k + 0.5, g.n - k + 0.5)
    ax.errorbar(g["T"], g.solve_rate,
                yerr=[np.maximum(0, g.solve_rate - a), np.maximum(0, b - g.solve_rate)],
                fmt="-o", color=col, capsize=2, label=c)
ax.axhline(0.90, color="0.5", ls="--", lw=1)
ax.set_xlabel("DD depth T"); ax.set_ylabel("solve_rate")
ax.set_title("rho=0.60, realized density 0.583 in every bin")
ax.set_xticks([2, 3, 4, 5]); ax.legend(fontsize=9)
fig.tight_layout()

**The curves cross.** The permuted arm is *better* than sorted at T=2 and T=3 and *worse* at
T=4 and T=5. So the 1.5-point pool delta is not a uniform tax; it is a redistribution towards
the shallow end, with the deepest bin paying 0.827 -> 0.786. If that is real, the order helps
exactly where the reasoning is longest, which is the interesting version of the hypothesis --
but n=98 in the deepest bin cannot settle it, and nothing here licenses the claim.

`well_formed_rate` is 1.0 and `truncated_rate` 0.0 in every bin of both runs, so none of this
is a grammar or budget failure.

## 3. The other order axis is still dead flat

`sudoku_memorization_eval --n-ordering 400` re-scores the same graphs under five adjacency
orderings. `bycolor` *re-instates* finding 16's leak and still does not help.

In [ ]:
ords = om[om.part == "ordering"]
display(ords[["ordering", "heldout_solve_rate", "gap", "p", "well_formed_rate",
              "n_instances"]].round(4))
print("identity is the reference row; nothing resolves, on a model trained WITHOUT the")
print("event order -- so removing one order convention did not make it fall back on the other.")

## 4. Memorization gap (pre-registration d)

Finding 17 measured +0.1000 at this budget (0.9567 own-training vs 0.8567 held out, p=2.6e-09).
Pre-registration (d) said the gap would **not** shrink below +0.1000.

In [ ]:
m = om[om.part == "memorization"].iloc[0]
display(om[om.part == "memorization"][["train_solve_rate", "heldout_solve_rate", "gap", "p",
                                       "n_instances"]].round(6))

# Is the shrink from +0.1000 to +0.0817 resolvable? Difference in differences on four rates.
# Independent binomial SEs are conservative here: both cells trained on the SAME 80000
# instances, so the two gaps are positively correlated and the true SE is smaller.
g_prev, g_now = 0.1000, float(m.gap)
se_prev = np.sqrt(0.9567 * 0.0433 / 600 + 0.8567 * 0.1433 / 600)
se_now = np.sqrt(m.train_solve_rate * (1 - m.train_solve_rate) / m.n_instances
                 + m.heldout_solve_rate * (1 - m.heldout_solve_rate) / m.n_instances)
z = (g_now - g_prev) / np.sqrt(se_prev ** 2 + se_now ** 2)
print(f"finding 17 gap {g_prev:+.4f} (SE {se_prev:.5f}) vs this run {g_now:+.4f} "
      f"(SE {se_now:.5f})")
print(f"difference {g_now - g_prev:+.4f}, z = {z:+.2f}, p = {2 * stats.norm.sf(abs(z)):.2f}")
print("-> nominally below +0.1000, but the shrink is not resolvable: a tie.")

## 5. Pre-registrations, as resolved

Written into `scripts/slurm/sudoku2_n24_eventorder.sbatch` before the job ran.

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | `solve_rate` falls resolvably but stays above 0.75 | **FALSIFIED** (toward "no change") | 0.8393 vs 0.8540, -0.0147, p=0.2253. The named falsifier -- "the order was pure decoration" -- holds. |
| (b) | `val_nll` rises by about the irreducible floor and no more | **CONFIRMED** | +0.0323 vs floor +0.0270; accuracy -0.0142 vs floor -0.0163; `n_params` and `max_len` unmoved |
| (c) | The depth cost survives: r at rho=0.60 stays below -0.8 | **CONFIRMED** | r = -0.979, and the T=2..5 drop is nearly twice the sorted arm's (0.203 vs 0.119) |
| (d) | The memorization gap does not shrink below +0.1000 | **NOMINALLY VIOLATED**, not resolvably | +0.0817 (p=5e-06 against its own held-out arm), but z=-0.76, p=0.45 against +0.1000 |

**(a) is a falsified prediction I am glad about, which is its own risk.** I predicted a
resolvable fall, and pre-registered what a null would mean so I could not reinterpret it
afterwards: *"the ascending order was pure decoration, the `min_unknown` shortcut was never
being used, and the open problem closes with the leak-free numbers intact."* That is the
reading I owe, and it is the one recorded here.

## 6. What this settles

- **The open problem finding 21 raised is closed.** `sudoku_trace.py:83` emits `sorted(...)` per
  round, so the deductions arrive in ascending node index for 100% of rounds, and
  `scripts/order_baseline_next_node.py` showed a hand rule with no deduction in it
  (`min_unknown`) scores 0.632 on `next_node` -- above the best SAE feature's 0.545. The
  shortcut was available; this run shows the model was not leaning on it.
- Finding 17's leak-free `solve_rate`s stand as deduction scores, not partly order-decoding
  scores, which was the specific worry this run was built to answer.
- Both order conventions in the data are now controlled by a retrain: `shuffle_adj` (finding 16)
  changed the answer dramatically, `shuffle_events` does not change it at all.

## 7. What it does not settle

- **A null is not a zero.** 139 won against 161 lost is a 1.5-point point estimate in the sorted
  arm's favour, and the paired interval admits a cost of up to ~3.7 points.
- **The crossing at T=4** (section 2) is unresolved and is the interesting residual.
- **The entropy confound, stated up front in the sbatch**: the permuted arm's training
  distribution has strictly higher entropy at the same `n_train`, so "lost a crutch" and "got
  less usable supervision per instance" are not separated here. Job 49705064 has since shown
  this task *is* data-bound at 80000 instances, which promotes that confound from a formality to
  a live alternative explanation for the whole -0.0147.
- One seed.

## 8. Next

1. Re-run this control at the 240000-instance budget 49705064 established. If both arms then sit
   at 1.000 the question is closed for good; if the permuted arm alone falls short, the entropy
   confound was the whole delta.
2. A depth-targeted pool (rho=0.55, n>=400 at T>=5) to test the crossing directly.
3. Re-score the SAE `next_node` features on *this* checkpoint: a model trained without the order
   should not have them at all, which is a sharp prediction finding 21 can test.